# Born Matrix Analysis

Load the maintained `16x30`, `nshell=1` pure-state covariance snapshot cache, rebuild the ordered OW basis, compute the Born matrix

\[
P_{ij} = \chi_i^\dagger C \chi_j, \qquad C = \tfrac12(G + I),
\]

for every saved sample and snapshot, and plot eigenvalue histograms per snapshot.

The OW index ordering used here is

\[
i = \text{trial\_orbital} + 2\,\text{unit\_cell} + 2N_xN_y\,\text{band},
\]

with minus band first and plus band second.


## Environment Setup and Imports

Configure CPU affinity, import the analysis stack, and resolve the repository source path.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# User-set CPU affinity range, inclusive.
CPU_RANGE = (0, 10)


def configure_cpu_affinity(cpu_range: tuple[int, int]) -> list[int] | None:
    start_cpu, end_cpu = cpu_range
    if end_cpu < start_cpu:
        raise ValueError(f"CPU_RANGE must satisfy start <= end, got {cpu_range}")

    cpu_list = list(range(int(start_cpu), int(end_cpu) + 1))
    thread_count = len(cpu_list)

    os.environ.setdefault("OMP_NUM_THREADS", str(thread_count))
    os.environ.setdefault("OPENBLAS_NUM_THREADS", str(thread_count))
    os.environ.setdefault("MKL_NUM_THREADS", str(thread_count))
    os.environ.setdefault("NUMEXPR_NUM_THREADS", str(thread_count))
    os.environ.setdefault("VECLIB_MAXIMUM_THREADS", str(thread_count))

    if hasattr(os, "sched_setaffinity"):
        try:
            os.sched_setaffinity(0, set(cpu_list))
            return sorted(os.sched_getaffinity(0))
        except OSError as exc:
            print(f"warning: could not set CPU affinity to {cpu_range}: {exc}")
            if hasattr(os, "sched_getaffinity"):
                return sorted(os.sched_getaffinity(0))
            return None

    print("warning: os.sched_setaffinity is unavailable on this platform")
    return None


ASSIGNED_CPUS = configure_cpu_affinity(CPU_RANGE)
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing src/fgtn/classA_U1FGTN.py")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from fgtn.classA_U1FGTN import classA_U1FGTN

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=6, suppress=True)

print(f"repo root: {REPO_ROOT}")
print(f"requested CPU range: {CPU_RANGE}")
print(f"assigned CPUs: {ASSIGNED_CPUS}")


## Cached Run Resolution and Matching OW Basis

Resolve the maintained `N16x30`, `nshell=1` snapshot cache, validate the manifest, and rebuild the OW basis that matches the cached simulation parameters.


In [ ]:
GPU_DATA_ROOT = REPO_ROOT / "colab_small_system_testing" / "gpu_data"
CACHE_ROOT = GPU_DATA_ROOT / "pure_state_covariance_snapshots"
EXPECTED = {
    "Nx": 16,
    "Ny": 30,
    "nshell": 1,
    "trial_orbitals": "X",
    "dw_truncation": True,
    "perfect_correction": True,
    "init_mode": "default",
    "snapshot_cycles": [5, 10, 20, 50],
    "samples": 10,
    "cycles": 50,
    "dtype": "c128",
}


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def resolve_run_dir() -> Path:
    latest_path = CACHE_ROOT / "runs" / "N16x30_nsh1_perfect_correction" / "latest_run.json"
    if latest_path.exists():
        latest_payload = load_json(latest_path)
        rel = latest_payload.get("run_dir_relative")
        if rel:
            run_dir = GPU_DATA_ROOT / rel
            if run_dir.exists():
                return run_dir
        summary_rel = latest_payload.get("run_summary_path_relative")
        if summary_rel:
            summary_path = GPU_DATA_ROOT / summary_rel
            if summary_path.exists():
                summary = load_json(summary_path)
                run_dir = Path(summary["run_dir"])
                if not run_dir.is_absolute():
                    run_dir = GPU_DATA_ROOT / run_dir
                if run_dir.exists():
                    return run_dir

    fallback = sorted((CACHE_ROOT / "runs" / "N16x30_nsh1_perfect_correction").glob("run_*"))
    if not fallback:
        raise FileNotFoundError("Could not resolve the N16x30_nsh1_perfect_correction snapshot run")
    return fallback[-1]


RUN_DIR = resolve_run_dir()
MANIFEST_PATH = RUN_DIR / "manifest.json"
MANIFEST = load_json(MANIFEST_PATH)
CFG = MANIFEST["config"]
SHARD_META = MANIFEST["shards"]["0"]
SHARD_PATH = RUN_DIR / SHARD_META["filename"]

assert int(CFG["Nx"]) == EXPECTED["Nx"]
assert int(CFG["Ny"]) == EXPECTED["Ny"]
assert int(CFG["nshell"]) == EXPECTED["nshell"]
assert str(CFG["trial_orbitals"]).upper() == EXPECTED["trial_orbitals"]
assert bool(CFG["dw_truncation"]) is EXPECTED["dw_truncation"]
assert bool(CFG["perfect_correction"]) is EXPECTED["perfect_correction"]
assert str(CFG["init_mode"]) == EXPECTED["init_mode"]
assert [int(x) for x in CFG["snapshot_cycles"]] == EXPECTED["snapshot_cycles"]
assert int(CFG["samples"]) == EXPECTED["samples"]
assert int(CFG["cycles"]) == EXPECTED["cycles"]
assert str(CFG["dtype"]) == EXPECTED["dtype"]
assert tuple(SHARD_META["shape"]) == (10, 4, 960, 960)
assert SHARD_PATH.exists(), SHARD_PATH

SNAPSHOT_CYCLES = [int(x) for x in CFG["snapshot_cycles"]]
NX = int(CFG["Nx"])
NY = int(CFG["Ny"])
NSHELL = int(CFG["nshell"])
N_CELLS = NX * NY
NLAYER = 2 * N_CELLS
BASIS_DIM = 4 * N_CELLS
ALPHA_TOP = float(CFG["alpha_top"])
ALPHA_TRIV = float(CFG["alpha_triv"])

print(f"run dir: {RUN_DIR}")
print(f"manifest: {MANIFEST_PATH}")
print(f"shard: {SHARD_PATH}")
print(f"snapshot cycles: {SNAPSHOT_CYCLES}")
print(f"expected Born matrix dimension: {BASIS_DIM} x {BASIS_DIM}")


In [ ]:
model = classA_U1FGTN(
    Nx=NX,
    Ny=NY,
    DW=True,
    nshell=NSHELL,
    filling_frac=float(CFG["filling_frac"]),
    alpha_1=ALPHA_TOP,
    alpha_2=ALPHA_TRIV,
    trial_orbitals=CFG["trial_orbitals"],
    dw_truncation=bool(CFG["dw_truncation"]),
)
model.construct_OW_projectors(
    nshell=NSHELL,
    DW=True,
    trial_orbitals=CFG["trial_orbitals"],
    dw_truncation=bool(CFG["dw_truncation"]),
)


def build_ordered_ow_basis(cpu_model: classA_U1FGTN) -> np.ndarray:
    columns = []

    for y in range(cpu_model.Ny):
        for x in range(cpu_model.Nx):
            columns.append(np.array(cpu_model.WF_Am[:, x, y], dtype=np.complex128, copy=True))
            columns.append(np.array(cpu_model.WF_Bm[:, x, y], dtype=np.complex128, copy=True))

    for y in range(cpu_model.Ny):
        for x in range(cpu_model.Nx):
            columns.append(np.array(cpu_model.WF_Ap[:, x, y], dtype=np.complex128, copy=True))
            columns.append(np.array(cpu_model.WF_Bp[:, x, y], dtype=np.complex128, copy=True))

    return np.column_stack(columns)


W = build_ordered_ow_basis(model)
W_dag = W.conj().T
OVERLAP = W_dag @ W

assert W.shape == (NLAYER, BASIS_DIM)
np.testing.assert_allclose(W[:, 0], model.WF_Am[:, 0, 0])
np.testing.assert_allclose(W[:, 1], model.WF_Bm[:, 0, 0])
np.testing.assert_allclose(W[:, 2 * N_CELLS], model.WF_Ap[:, 0, 0])
np.testing.assert_allclose(W[:, 2 * N_CELLS + 1], model.WF_Bp[:, 0, 0])

col_norms = np.linalg.norm(W, axis=0)
np.testing.assert_allclose(col_norms, np.ones_like(col_norms), atol=1e-10)

print(f"W shape: {W.shape}")
print(f"overlap shape: {OVERLAP.shape}")
print(f"DW locations: {model.DW_loc}")


## `C = I` Overlap Diagnostics

Use the identity two-point function so that `P = W^\dagger W`, then inspect the overlap eigenspectrum, rank, and zero-mode count directly.


In [ ]:
identity_phys = np.eye(NLAYER, dtype=np.complex128)
IDENTITY_RANK_TOL = 1e-10

P_identity = W_dag @ identity_phys @ W
P_identity = 0.5 * (P_identity + P_identity.conj().T)
eigvals_identity = np.linalg.eigvalsh(P_identity)
identity_herm_err = float(np.max(np.abs(P_identity - P_identity.conj().T)))
identity_rank = int(np.count_nonzero(eigvals_identity > IDENTITY_RANK_TOL))
identity_zero_mode_count = int(eigvals_identity.size - identity_rank)
identity_hist_counts, identity_hist_edges = np.histogram(eigvals_identity, bins=80)

print("C = I implies P_identity = W^dagger W")
print(f"C = I Born matrix shape: {P_identity.shape}")
print(f"C = I Hermiticity error: {identity_herm_err:.3e}")
print(f"C = I eigval range: [{eigvals_identity.min():.6f}, {eigvals_identity.max():.6f}]")
print(f"C = I total eigenvalues: {eigvals_identity.size}")
print(f"C = I histogram bin-count sum: {int(identity_hist_counts.sum())}")
print(f"rank(W^dagger W): {identity_rank}")
print(f"rank tolerance: {IDENTITY_RANK_TOL:.1e}")
print(f"inferred zero modes: {identity_zero_mode_count}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(np.sort(eigvals_identity), linewidth=1.25)
axes[0].set_xlabel("sorted index")
axes[0].set_ylabel("eigenvalue")
axes[0].set_title(r"Sorted eig($W^\dagger W$) for $C = I$")

axes[1].hist(eigvals_identity, bins=identity_hist_edges, color="tab:orange", alpha=0.85, edgecolor="white")
axes[1].axvline(0.0, color="black", linewidth=1.0, alpha=0.5)
axes[1].set_title(r"Histogram of eig($W^\dagger W$) for $C = I$")
axes[1].set_xlabel("eigenvalue")
axes[1].set_ylabel("bin count")

fig.tight_layout()
plt.show()


## Cached Born-Matrix Spectra Across All Samples

Compute `P = W^\dagger C W` for every cached sample and snapshot, then summarize and visualize the pooled eigenspectra.


In [ ]:
shard = np.load(SHARD_PATH, mmap_mode="r", allow_pickle=False)
assert shard.shape == (EXPECTED["samples"], len(SNAPSHOT_CYCLES), NLAYER, NLAYER)
assert shard.dtype == np.complex128

eigvals_by_snapshot = np.empty((len(SNAPSHOT_CYCLES), EXPECTED["samples"], BASIS_DIM), dtype=np.float64)

sample0_snapshot0_C_herm_err = None
sample0_snapshot0_P_herm_err = None

for snap_idx, cycle in enumerate(tqdm(SNAPSHOT_CYCLES, desc="Snapshots")):
    for sample_idx in tqdm(range(EXPECTED["samples"]), desc=f"cycle {cycle}", leave=False):
        G = np.asarray(shard[sample_idx, snap_idx], dtype=np.complex128)
        C = 0.5 * (G + identity_phys)
        if sample0_snapshot0_C_herm_err is None:
            sample0_snapshot0_C_herm_err = float(np.max(np.abs(C - C.conj().T)))
        P = W_dag @ C @ W
        P = 0.5 * (P + P.conj().T)
        if sample0_snapshot0_P_herm_err is None:
            sample0_snapshot0_P_herm_err = float(np.max(np.abs(P - P.conj().T)))
        eigvals_by_snapshot[snap_idx, sample_idx] = np.linalg.eigvalsh(P)

assert sample0_snapshot0_C_herm_err is not None
assert sample0_snapshot0_P_herm_err is not None
assert sample0_snapshot0_C_herm_err < 1e-10
assert sample0_snapshot0_P_herm_err < 1e-10

print(f"eigenvalue array shape: {eigvals_by_snapshot.shape}")
print(f"C hermiticity error on first sample/snapshot: {sample0_snapshot0_C_herm_err:.3e}")
print(f"P hermiticity error on first sample/snapshot: {sample0_snapshot0_P_herm_err:.3e}")


In [ ]:
all_eigvals = eigvals_by_snapshot.reshape(-1)
eig_min = float(all_eigvals.min())
eig_max = float(all_eigvals.max())
num_bins = 80
bins = np.linspace(eig_min, eig_max, num_bins + 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, cycle, eigvals in zip(axes, SNAPSHOT_CYCLES, eigvals_by_snapshot):
    pooled = eigvals.reshape(-1)
    ax.hist(pooled, bins=bins, color="tab:blue", alpha=0.8, edgecolor="white")
    ax.set_title(f"Cycle {cycle} | pooled over {eigvals.shape[0]} samples")
    ax.set_xlabel("eigenvalue")
    ax.set_ylabel("count")
    ax.axvline(0.0, color="black", linewidth=1.0, alpha=0.5)

fig.suptitle(
    f"Born-matrix eigenvalue histograms | N={NX}x{NY}, nshell={NSHELL}, basis dim={BASIS_DIM}",
    fontsize=14,
)
fig.tight_layout()
plt.show()


In [ ]:
summary = {
    "run_dir": str(RUN_DIR),
    "manifest": str(MANIFEST_PATH),
    "shard": str(SHARD_PATH),
    "snapshot_cycles": SNAPSHOT_CYCLES,
    "W_shape": W.shape,
    "Born_matrix_shape": (BASIS_DIM, BASIS_DIM),
    "eigvals_shape": eigvals_by_snapshot.shape,
    "eig_min": float(eigvals_by_snapshot.min()),
    "eig_max": float(eigvals_by_snapshot.max()),
}
summary


## Cached Covariance Snapshot Spectra With Matching OW Basis

This section uses the same OW basis configuration as the cached run metadata in `CFG`, builds `P = W^\dagger C W` for each saved snapshot, and plots one sorted-eigenspectrum subplot per snapshot.


In [ ]:
CACHE_SNAPSHOT_PLOT_MODE = "sample_average"  # "sample_average" or "sample_index"
CACHE_PLOT_SAMPLE_INDEX = 0
CACHE_SHOW_OVERLAP_REFERENCE = True
CACHE_SORTED_MARKER = "."
CACHE_SORTED_MARKERSIZE = 3.0

print("Using cached OW configuration from manifest:")
print(f"  trial_orbitals: {CFG['trial_orbitals']}")
print(f"  nshell: {CFG['nshell']}")
print(f"  DW: {CFG['DW']}")
print(f"  dw_truncation: {CFG['dw_truncation']}")
print(f"  alpha_top/alpha_triv: ({CFG['alpha_top']}, {CFG['alpha_triv']})")
print(f"  plot mode: {CACHE_SNAPSHOT_PLOT_MODE}")


In [ ]:
def cached_G_for_snapshot(snap_idx: int, *, mode: str, sample_index: int) -> tuple[np.ndarray, str]:
    if mode == "sample_average":
        G_snap = np.asarray(np.mean(shard[:, snap_idx], axis=0), dtype=np.complex128)
        label = f"sample-average over {shard.shape[0]} cached samples"
        return G_snap, label
    if mode == "sample_index":
        if not (0 <= sample_index < shard.shape[0]):
            raise IndexError(f"sample_index must be in [0, {shard.shape[0] - 1}], got {sample_index}")
        G_snap = np.asarray(shard[sample_index, snap_idx], dtype=np.complex128)
        label = f"cached sample index {sample_index}"
        return G_snap, label
    raise ValueError(f"Unsupported CACHE_SNAPSHOT_PLOT_MODE={mode!r}")


cached_snapshot_plot_records = []
for snap_idx, cycle in enumerate(SNAPSHOT_CYCLES):
    G_snap, covariance_source_label = cached_G_for_snapshot(
        snap_idx,
        mode=CACHE_SNAPSHOT_PLOT_MODE,
        sample_index=CACHE_PLOT_SAMPLE_INDEX,
    )
    C_snap = 0.5 * (G_snap + identity_phys)
    C_snap = 0.5 * (C_snap + C_snap.conj().T)
    P_snap = W_dag @ C_snap @ W
    P_snap = 0.5 * (P_snap + P_snap.conj().T)
    eigs_snap = np.linalg.eigvalsh(P_snap)
    hist_counts_snap, _ = np.histogram(eigs_snap, bins=80)
    cached_snapshot_plot_records.append(
        {
            "snap_idx": snap_idx,
            "cycle": cycle,
            "covariance_source_label": covariance_source_label,
            "G": G_snap,
            "C": C_snap,
            "P": P_snap,
            "eigs": eigs_snap,
            "eig_min": float(eigs_snap.min()),
            "eig_max": float(eigs_snap.max()),
            "hist_count_sum": int(hist_counts_snap.sum()),
        }
    )

for rec in cached_snapshot_plot_records:
    print(
        f"cycle {rec['cycle']}: source={rec['covariance_source_label']}, "
        f"num eigs={rec['eigs'].size}, hist sum={rec['hist_count_sum']}, "
        f"range=[{rec['eig_min']:.6f}, {rec['eig_max']:.6f}]"
    )


In [ ]:
n_snapshots = len(cached_snapshot_plot_records)
ncols = 2
nrows = int(np.ceil(n_snapshots / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.8 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

sorted_overlap_reference = np.sort(eigvals_identity)

for ax, rec in zip(axes, cached_snapshot_plot_records):
    sorted_snapshot_eigs = np.sort(rec['eigs'])
    ax.plot(
        np.arange(sorted_snapshot_eigs.size),
        sorted_snapshot_eigs,
        CACHE_SORTED_MARKER,
        markersize=CACHE_SORTED_MARKERSIZE,
        label=r"eig($W^\dagger C W$)",
    )
    if CACHE_SHOW_OVERLAP_REFERENCE:
        ax.plot(
            np.arange(sorted_overlap_reference.size),
            sorted_overlap_reference,
            linestyle='--',
            linewidth=1.0,
            alpha=0.85,
            label=r"eig($W^\dagger W$)",
        )
    ax.set_title(f"Cycle {rec['cycle']}")
    ax.set_xlabel("sorted index")
    ax.set_ylabel("eigenvalue")
    ax.legend(fontsize=8, loc='best')

for ax in axes[n_snapshots:]:
    ax.axis('off')

fig.suptitle(
    f"Sorted eig(W^dagger C W) for cached covariance snapshots | mode={CACHE_SNAPSHOT_PLOT_MODE}",
    fontsize=14,
)
fig.tight_layout()
plt.show()


## Configurable OW Basis and User-Provided `C` Study

This section lets you vary `nshell`, `DW`, `dw_truncation`, `alpha_1`, `alpha_2`, and `trial_orbitals`, rebuild the same ordered OW basis, and study the eigenspectrum of `W^\dagger W` together with its numerical rank.

Recall that the nonzero eigenvalues of `W^\dagger W` are the squares of the singular values of `W`. For `N=16x30`, `W` has shape `(960, 1920)`, so `W^\dagger W` has shape `(1920, 1920)` and, in the ideal rank-`960` case, contains `960` nonzero eigenvalues and `960` zero modes.


### Sweep Setup

Set the OW-basis parameter ranges and define the helper that constructs `W`, `W^\dagger W`, and the overlap-rank summary for each case.


In [ ]:
import itertools

import pandas as pd


In [ ]:
SWEEP_NX = 16
SWEEP_NY = 30
SWEEP_FILLING_FRAC = 0.5
SWEEP_TRIAL_ORBITALS = ["X"]
SWEEP_NSHELL_VALUES = [None, 0, 1, 2]
SWEEP_DW_OPTIONS = [True, False]
SWEEP_DW_TRUNCATION_OPTIONS = [False, True]
SWEEP_ALPHA_PAIRS = [
    (1.0, 30.0),
    # (1.0, 10.0),
    # (0.5, 30.0),
]
SKIP_INCONSISTENT_CASES = True
SWEEP_OVERLAP_RANK_TOL = 1e-10
OVERLAP_HIST_BINS = 80
COMPARE_HIST_BINS = 100
PLOT_OVERLAP_SORTED = True
PLOT_OVERLAP_HIST = True
MAX_PLOT_POINTS = None


def make_model(
    *,
    nx: int,
    ny: int,
    nshell,
    dw: bool,
    alpha_1: float,
    alpha_2: float,
    trial_orbitals: str,
    dw_truncation: bool,
    filling_frac: float = 0.5,
) -> classA_U1FGTN:
    model = classA_U1FGTN(
        Nx=nx,
        Ny=ny,
        DW=dw,
        nshell=nshell,
        filling_frac=filling_frac,
        alpha_1=alpha_1,
        alpha_2=alpha_2,
        trial_orbitals=trial_orbitals,
        dw_truncation=dw_truncation,
    )
    model.construct_OW_projectors(
        nshell=nshell,
        DW=dw,
        trial_orbitals=trial_orbitals,
        dw_truncation=dw_truncation,
    )
    return model


def overlap_record(
    *,
    nx: int,
    ny: int,
    nshell,
    dw: bool,
    alpha_1: float,
    alpha_2: float,
    trial_orbitals: str,
    dw_truncation: bool,
):
    model = make_model(
        nx=nx,
        ny=ny,
        nshell=nshell,
        dw=dw,
        alpha_1=alpha_1,
        alpha_2=alpha_2,
        trial_orbitals=trial_orbitals,
        dw_truncation=dw_truncation,
        filling_frac=SWEEP_FILLING_FRAC,
    )

    W_case = build_ordered_ow_basis(model)
    overlap = W_case.conj().T @ W_case
    overlap = 0.5 * (overlap + overlap.conj().T)
    overlap_eigs = np.linalg.eigvalsh(overlap)

    nlayer = 2 * nx * ny
    basis_dim = 4 * nx * ny
    assert W_case.shape == (nlayer, basis_dim)
    np.testing.assert_allclose(np.linalg.norm(W_case, axis=0), np.ones((basis_dim,)), atol=1e-10)
    assert overlap.shape == (basis_dim, basis_dim)
    assert overlap_eigs.size == basis_dim

    overlap_rank_tol = float(SWEEP_OVERLAP_RANK_TOL)
    overlap_rank = int(np.count_nonzero(overlap_eigs > overlap_rank_tol))
    overlap_zero_mode_count = int(overlap_eigs.size - overlap_rank)

    return {
        "Nx": nx,
        "Ny": ny,
        "nshell": nshell,
        "DW": dw,
        "dw_truncation": dw_truncation,
        "alpha_1": alpha_1,
        "alpha_2": alpha_2,
        "trial_orbitals": trial_orbitals,
        "W_shape": W_case.shape,
        "overlap_dim": int(overlap_eigs.size),
        "overlap_eval_min": float(overlap_eigs.min()),
        "overlap_eval_max": float(overlap_eigs.max()),
        "overlap_rank_tol": overlap_rank_tol,
        "overlap_rank": overlap_rank,
        "overlap_zero_mode_count": overlap_zero_mode_count,
        "DW_loc": getattr(model, "DW_loc", None),
        "overlap": overlap,
        "overlap_eigs": overlap_eigs,
        "W": W_case,
        "model": model,
    }


### Sweep Execution and Case Summary

Run the configurable sweep, collect the overlap-spectrum records, and inspect one selected case numerically.


In [ ]:
records = []

for trial_orbitals, nshell, dw, dw_truncation, (alpha_1, alpha_2) in itertools.product(
    SWEEP_TRIAL_ORBITALS,
    SWEEP_NSHELL_VALUES,
    SWEEP_DW_OPTIONS,
    SWEEP_DW_TRUNCATION_OPTIONS,
    SWEEP_ALPHA_PAIRS,
):
    if SKIP_INCONSISTENT_CASES and (not dw) and dw_truncation:
        continue

    rec = overlap_record(
        nx=SWEEP_NX,
        ny=SWEEP_NY,
        nshell=nshell,
        dw=dw,
        alpha_1=alpha_1,
        alpha_2=alpha_2,
        trial_orbitals=trial_orbitals,
        dw_truncation=dw_truncation,
    )
    records.append(rec)

summary_df = pd.DataFrame(
    [
        {k: v for k, v in rec.items() if k not in {"overlap", "overlap_eigs", "W", "model"}}
        for rec in records
    ]
)

summary_df


In [ ]:
case_idx = 0
rec = records[case_idx]
case_hist_counts, _ = np.histogram(rec["overlap_eigs"], bins=OVERLAP_HIST_BINS)

print("Case:")
for key in ["Nx", "Ny", "nshell", "DW", "dw_truncation", "alpha_1", "alpha_2", "trial_orbitals", "W_shape", "DW_loc"]:
    print(f"  {key}: {rec[key]}")

print(f"\ntotal number of overlap eigenvalues: {rec['overlap_dim']}")
print(f"histogram bin-count sum: {int(case_hist_counts.sum())}")
print(f"min/max eig(W^dagger W): {rec['overlap_eval_min']:.6f}, {rec['overlap_eval_max']:.6f}")
print(f"rank(W^dagger W): {rec['overlap_rank']}")
print(f"rank tolerance: {rec['overlap_rank_tol']:.1e}")
print(f"inferred zero modes: {rec['overlap_zero_mode_count']}")


### Sweep Plots

Plot the sorted overlap eigenspectra and overlap histograms for the configurable OW-basis cases.


In [ ]:
if PLOT_OVERLAP_SORTED:
    plt.figure(figsize=(11, 6))

    for rec in records:
        overlap_eigs = np.sort(rec["overlap_eigs"])
        if MAX_PLOT_POINTS is not None:
            idx = np.linspace(0, len(overlap_eigs) - 1, MAX_PLOT_POINTS).astype(int)
            x = idx
            y = overlap_eigs[idx]
        else:
            x = np.arange(len(overlap_eigs))
            y = overlap_eigs

        label = (
            f"trial={rec['trial_orbitals']}, "
            f"nshell={rec['nshell']}, "
            f"DW={rec['DW']}, "
            f"dwtrunc={rec['dw_truncation']}, "
            f"a=({rec['alpha_1']},{rec['alpha_2']})"
        )
        plt.plot(x, y, label=label, linewidth=1.5)

    plt.xlabel("sorted index")
    plt.ylabel("eigenvalue")
    plt.title(rf"Sorted eig($W^\dagger W$) for ordered OW basis, N={SWEEP_NX}x{SWEEP_NY}")
    plt.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


In [ ]:
if PLOT_OVERLAP_HIST:
    plt.figure(figsize=(10, 6))
    for rec in records:
        plt.hist(
            rec["overlap_eigs"],
            bins=OVERLAP_HIST_BINS,
            histtype="step",
            linewidth=1.5,
            label=(
                f"trial={rec['trial_orbitals']}, "
                f"nshell={rec['nshell']}, "
                f"DW={rec['DW']}, "
                f"dwtrunc={rec['dw_truncation']}, "
                f"a=({rec['alpha_1']},{rec['alpha_2']})"
            ),
        )
    plt.xlabel("eigenvalue")
    plt.ylabel("bin count")
    plt.title(rf"Histogram of eig($W^\dagger W$) for ordered OW basis, N={SWEEP_NX}x{SWEEP_NY}")
    plt.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


## User-Supplied Covariance / Born Matrix Comparison

For a selected configurable OW-basis case, this section either uses a cached covariance snapshot or a user-provided top-layer two-point function `C_user`, then computes `P = W^\dagger C W` and compares `eig(P)` directly against `eig(W^\dagger W)` for the same case.


### Comparison Setup

Choose the configurable OW-basis case, select either a cached covariance or a custom `C_user`, and build the corresponding Born matrix.


In [ ]:
COMPARE_CASE_INDEX = 0
USE_CACHE_SNAPSHOT = True
CACHE_SAMPLE_INDEX = 0
CACHE_SNAPSHOT_INDEX = 0

compare_rec = records[COMPARE_CASE_INDEX]
W_compare = compare_rec["W"]
W_compare_dag = W_compare.conj().T
overlap_compare = compare_rec["overlap"]
overlap_eigvals_compare = compare_rec["overlap_eigs"]
overlap_rank_compare = compare_rec["overlap_rank"]
overlap_rank_tol_compare = compare_rec["overlap_rank_tol"]
overlap_zero_modes_compare = compare_rec["overlap_zero_mode_count"]
nlayer_compare = W_compare.shape[0]
basis_dim_compare = W_compare.shape[1]

# If you want to override the cache snapshot, set C_user to a (2*Nx*Ny, 2*Nx*Ny)
# complex Hermitian two-point matrix and set USE_CACHE_SNAPSHOT = False.
C_user = np.diag(np.concatenate([np.ones(nlayer_compare // 2), np.zeros(nlayer_compare // 2)]))

if USE_CACHE_SNAPSHOT:
    assert nlayer_compare == NLAYER, "Cache snapshots are only compatible with N=16x30 top-layer dimension 960."
    G_compare = np.asarray(shard[CACHE_SAMPLE_INDEX, CACHE_SNAPSHOT_INDEX], dtype=np.complex128)
    C_compare = 0.5 * (G_compare + np.eye(nlayer_compare, dtype=np.complex128))
    covariance_label = (
        f"cache snapshot: sample={CACHE_SAMPLE_INDEX}, cycle={SNAPSHOT_CYCLES[CACHE_SNAPSHOT_INDEX]}"
    )
else:
    if C_user is None:
        raise ValueError("Set C_user when USE_CACHE_SNAPSHOT is False.")
    C_compare = np.asarray(C_user, dtype=np.complex128)
    covariance_label = "user-supplied C"

assert C_compare.shape == (nlayer_compare, nlayer_compare)
C_compare = 0.5 * (C_compare + C_compare.conj().T)

P_compare = W_compare_dag @ C_compare @ W_compare
P_compare = 0.5 * (P_compare + P_compare.conj().T)
born_eigvals_compare = np.linalg.eigvalsh(P_compare)
born_hist_counts, _ = np.histogram(born_eigvals_compare, bins=COMPARE_HIST_BINS)
overlap_hist_counts_compare, _ = np.histogram(overlap_eigvals_compare, bins=COMPARE_HIST_BINS)

print(f"comparison case index: {COMPARE_CASE_INDEX}")
print(f"covariance source: {covariance_label}")
print(f"C shape: {C_compare.shape}")
print(f"P shape: {P_compare.shape}")
print(f"total overlap eigenvalues: {overlap_eigvals_compare.size}")
print(f"overlap histogram bin-count sum: {int(overlap_hist_counts_compare.sum())}")
print(f"rank(W^dagger W): {overlap_rank_compare}")
print(f"rank tolerance: {overlap_rank_tol_compare:.1e}")
print(f"inferred zero modes: {overlap_zero_modes_compare}")
print(f"total Born-matrix eigenvalues: {born_eigvals_compare.size}")
print(f"Born-matrix histogram bin-count sum: {int(born_hist_counts.sum())}")
print(f"eig(P) range: [{born_eigvals_compare.min():.6f}, {born_eigvals_compare.max():.6f}]")
print(f"eig(W^dagger W) range: [{overlap_eigvals_compare.min():.6f}, {overlap_eigvals_compare.max():.6f}]")


### Comparison Plots

Visualize the sorted eigenspectra and histogram comparison between `W^\dagger C W` and `W^\dagger W` for the selected case.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(np.sort(born_eigvals_compare), label=r"eig($W^\dagger C W$)", linewidth=1.5)
axes[0].plot(np.sort(overlap_eigvals_compare), label=r"eig($W^\dagger W$)", linewidth=1.5)
axes[0].set_xlabel("sorted index")
axes[0].set_ylabel("eigenvalue")
axes[0].set_title("Sorted spectra")
axes[0].legend()

bins_compare = np.linspace(
    min(born_eigvals_compare.min(), overlap_eigvals_compare.min()),
    max(born_eigvals_compare.max(), overlap_eigvals_compare.max()),
    COMPARE_HIST_BINS,
)
axes[1].hist(born_eigvals_compare, bins=bins_compare, histtype="step", linewidth=1.5, label=r"eig($W^\dagger C W$)")
axes[1].hist(overlap_eigvals_compare, bins=bins_compare, histtype="step", linewidth=1.5, label=r"eig($W^\dagger W$)")
axes[1].set_xlabel("eigenvalue")
axes[1].set_ylabel("bin count")
axes[1].set_title("Histogram comparison")
axes[1].legend()

fig.suptitle(f"Born-matrix comparison for case {COMPARE_CASE_INDEX}: {covariance_label}")
fig.tight_layout()
plt.show()
